In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import ElasticNet

df = pd.read_excel(r"C:\Users\HP\Desktop\EV_Fleet_City_Highway_25000.xlsx")
df
df['maintenance_status'] = df['maintenance_status'].fillna("Unknown")
df['total_km_run'] = df['total_km_run'].astype(int)
df['battery_percentage'] = df['battery_percentage'].astype(int)
df['speed_kmph'] = df['speed_kmph'].astype(int)
car_encoder = LabelEncoder()
road_encoder = LabelEncoder()
charging_encoder = LabelEncoder()
maintenance_encoder = LabelEncoder()

df['car_type'] = car_encoder.fit_transform(df['car_type'].astype(str))
df['road_type'] = road_encoder.fit_transform(df['road_type'].astype(str))
df['charging_status'] = charging_encoder.fit_transform(df['charging_status'].astype(str))
df['maintenance_status'] = maintenance_encoder.fit_transform(df['maintenance_status'].astype(str))
X = df[
    [
        'car_type',
        'battery_capacity_kwh',
        'max_range_km',
        'vehicle_weight_kg',
        'motor_power_kw',
        'battery_percentage',
        'road_type',
        'speed_kmph',
        'charging_status',
        'maintenance_status',
        'total_km_run',
        'passenger_count'
    ]
]
X
y = df['remaining_distance_km']
y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R² Score:", r2)

print("Train score:", model.score(X_train, y_train))
print("Test score:", model.score(X_test, y_test))

df_temp = df.copy()
df_temp['car_type_raw'] = car_encoder.inverse_transform(df['car_type'])

car_defaults = df_temp.groupby('car_type_raw')[[
    'battery_capacity_kwh',
    'max_range_km',
    'vehicle_weight_kg',
    'motor_power_kw'
]].mean().to_dict(orient='index')

def safe_encode(encoder, value):
    if value in encoder.classes_:
        return encoder.transform([value])[0]
    return 0
print("\n🚗 EV Remaining Distance Prediction\n")

total_km_run = int(input("Enter Total KM Run: "))
battery_percentage = int(input("Enter Battery Percentage: "))
speed_kmph = int(input("Enter Speed (kmph): "))

print("\nAvailable Car Types:")
for i, c in enumerate(car_encoder.classes_):
    print(f"{i}. {c}")

car_choice = int(input("Select Car Type: "))
car_type = car_encoder.classes_[car_choice]
car_encoded = car_encoder.transform([car_type])[0]

print("\nAvailable Road Types:")
for i, r in enumerate(road_encoder.classes_):
    print(f"{i}. {r}")

road_choice = int(input("Select Road Type: "))
road_type = road_encoder.classes_[road_choice]
road_encoded = road_encoder.transform([road_type])[0]
car_default = car_defaults[car_type]

battery_capacity_kwh = car_default['battery_capacity_kwh']
max_range_km = car_default['max_range_km']
vehicle_weight_kg = car_default['vehicle_weight_kg']
motor_power_kw = car_default['motor_power_kw']

charging_encoded = safe_encode(charging_encoder, "Fast")
maintenance_encoded = safe_encode(maintenance_encoder, "Low")

passenger_count = int(input("Enter Passenger Count: "))




input_data = np.array([[
    car_encoded,
    battery_capacity_kwh,
    max_range_km,
    vehicle_weight_kg,
    motor_power_kw,
    battery_percentage,
    road_encoded,
    speed_kmph,
    charging_encoded,
    maintenance_encoded,
    total_km_run,
    passenger_count
]])
input_df = pd.DataFrame(input_data, columns=X.columns)
input_scaled = scaler.transform(input_df)
prediction = model.predict(input_scaled)

print("\n🚗 Predicted Remaining Distance (KM):", round(prediction[0], 2))
import pickle
with open("ev_model.pkl", "wb") as file:
    pickle.dump(model, file)


Mean Squared Error: 1350.007017711198
R² Score: 0.8750813819930107
Train score: 0.8782309465973477
Test score: 0.8750813819930107

🚗 EV Remaining Distance Prediction


Available Car Types:
0. MG ZS EV
1. Tata Punch EV
2. Volvo EX40

Available Road Types:
0. City
1. Highway

🚗 Predicted Remaining Distance (KM): 13.17
